In [ ]:
!pip install -q huggingface_hub torch torchvision pillow opencv-python

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Folder where we will keep the soil model
MODEL_DIR = "/content/soil_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Download pretrained soil classification model
MODEL_PATH = hf_hub_download(
    repo_id="watermelon-elite/aaagriculture-models",
    filename="soil_type_model.pth",
    local_dir=MODEL_DIR
)

print("======================================")
print("SOIL MODEL DOWNLOADED SUCCESSFULLY")
print("======================================")
print("Model path:")
print(MODEL_PATH)
print()
print("File exists:", os.path.exists(MODEL_PATH))

soil_type_model.pth: reconstructing file:   0%|          |  0.00B /  136MB            

soil_type_model.pth: downloading bytes:           |  0.00B            

SOIL MODEL DOWNLOADED SUCCESSFULLY
Model path:
/content/soil_model/soil_type_model.pth

File exists: True


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# --------------------------------------------------
# Device
# --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

# --------------------------------------------------
# Soil classes
# --------------------------------------------------
SOIL_CLASSES = [
    "Sandy",
    "Clay",
    "Loamy",
    "Silt",
    "Peat",
    "Chalky"
]

# --------------------------------------------------
# Create ResNet18
# --------------------------------------------------
soil_model = models.resnet18(weights=None)

# Replace final layer for 6 soil classes
soil_model.fc = nn.Linear(
    soil_model.fc.in_features,
    len(SOIL_CLASSES)
)

# --------------------------------------------------
# Load pretrained checkpoint
# --------------------------------------------------
MODEL_PATH = "/content/soil_model/soil_type_model.pth"

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

print("Checkpoint loaded successfully.")
print("Checkpoint type:", type(checkpoint))

Using device: cpu
Checkpoint loaded successfully.
Checkpoint type: <class 'dict'>


In [ ]:
# ============================================================
# EXACT SOIL MODEL ARCHITECTURE
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# Get classes directly from checkpoint
# ------------------------------------------------------------

SOIL_CLASSES = checkpoint["class_names"]

print("Soil classes:")
for i, name in enumerate(SOIL_CLASSES):
    print(f"{i}: {name}")


# ------------------------------------------------------------
# Create ResNet18
# ------------------------------------------------------------

backbone = models.resnet18(weights=None)


# ------------------------------------------------------------
# EXACT classifier found in checkpoint
#
# backbone.fc.1.weight = (256, 512)
# backbone.fc.4.weight = (6, 256)
# ------------------------------------------------------------

backbone.fc = nn.Sequential(

    nn.Identity(),

    nn.Linear(512, 256),

    nn.ReLU(),

    nn.Dropout(p=0.5),

    nn.Linear(256, 6)
)


# ------------------------------------------------------------
# Put ResNet inside "backbone"
# ------------------------------------------------------------

class SoilClassifier(nn.Module):

    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone

    def forward(self, x):

        return self.backbone(x)


# Create final model
soil_model = SoilClassifier(backbone)


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

state_dict = checkpoint["model_state_dict"]

soil_model.load_state_dict(state_dict)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

soil_model = soil_model.to(DEVICE)

soil_model.eval()


# ------------------------------------------------------------
# SUCCESS
# ------------------------------------------------------------

print()
print("==============================================")
print("       SOIL MODEL LOADED SUCCESSFULLY")
print("==============================================")
print("Architecture : ResNet18")
print("Classifier   : 512 → 256 → 6")
print("Classes      :", SOIL_CLASSES)
print("Device       :", DEVICE)
print("Epoch        :", checkpoint["epoch"])
print("Validation Accuracy :", checkpoint["val_acc"])
print("==============================================")

Soil classes:
0: chalky
1: Clay soil
2: Loamy
3: Peat Soil
4: Sandy soil
5: Silt

       SOIL MODEL LOADED SUCCESSFULLY
Architecture : ResNet18
Classifier   : 512 → 256 → 6
Classes      : ['chalky', 'Clay soil', 'Loamy', 'Peat Soil', 'Sandy soil', 'Silt']
Device       : cpu
Epoch        : 24
Validation Accuracy : 99.37888198757764


In [ ]:
# ============================================================
# LIVE SOIL CLASSIFICATION + FINAL SUMMARY REPORT
# ============================================================

import IPython
from google.colab.output import eval_js
from base64 import b64decode
import io
import time
import torch
import pandas as pd

from PIL import Image, ImageDraw
from IPython.display import display, HTML

# ============================================================
# 1. CHECK SOIL MODEL
# ============================================================

if "soil_model" not in globals():
    raise RuntimeError("❌ soil_model is not loaded.")

if "soil_transform" not in globals():
    raise RuntimeError("❌ soil_transform is not loaded.")

if "SOIL_CLASSES" not in globals():
    raise RuntimeError("❌ SOIL_CLASSES is not loaded.")

# ============================================================
# 2. WEBCAM HTML
# ============================================================

VIDEO_HTML = """
<div style="text-align:center;">
    <video id="webcam"
           autoplay
           playsinline
           width="640"
           height="480"
           style="
           border:3px solid black;
           border-radius:10px;">
    </video>
    <br><br>
    <button onclick="stopWebcam()"
            style="
            font-size:18px;
            padding:10px 25px;
            border-radius:8px;
            border:none;
            cursor:pointer;">
        🛑 Stop Webcam
    </button>
</div>
"""

# ============================================================
# 3. WEBCAM JAVASCRIPT
# ============================================================

VIDEO_JS = """
(async () => {
    const video = document.getElementById("webcam");
    if (!video) {
        throw new Error("Webcam video element not found.");
    }
    if (window.webcamStream) {
        window.webcamStream.getTracks().forEach(track => track.stop());
    }
    const stream = await navigator.mediaDevices.getUserMedia({
        video: true,
        audio: false
    });
    video.srcObject = stream;
    window.webcamStream = stream;
    window.webcamRunning = true;
    window.isWebcamActive = function() {
        return window.webcamRunning === true;
    };
    window.captureFrame = function() {
        if (!window.webcamRunning) {
            return null;
        }
        const canvas = document.createElement("canvas");
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        const ctx = canvas.getContext("2d");
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        return canvas.toDataURL("image/jpeg", 0.80);
    };
    window.stopWebcam = function() {
        window.webcamRunning = false;
        if (window.webcamStream) {
            window.webcamStream.getTracks().forEach(track => track.stop());
        }
        video.srcObject = null;
    };
})();
"""

# ============================================================
# 4. START WEBCAM
# ============================================================

IPython.display.clear_output(wait=True)
display(HTML(VIDEO_HTML))
eval_js(VIDEO_JS)
print("Starting webcam...")
print("Please allow camera access if requested.")
print("")

# ============================================================
# 5. WAIT FOR CAMERA
# ============================================================

camera_ready = False
for _ in range(30):
    try:
        camera_ready = eval_js("window.isWebcamActive()")
    except Exception:
        camera_ready = False
    if camera_ready:
        break
    time.sleep(0.5)

if not camera_ready:
    print("❌ Webcam could not be started.")
else:
    print("✅ Webcam successfully started.")
    print("")
    print("🌱 LIVE SOIL CLASSIFICATION RUNNING")
    print("")

    # ========================================================
    # SESSION TRACKING
    # ========================================================
    soil_counts = {}
    soil_confidences = {}
    total_frames = 0
    confidence_values = []

    # ========================================================
    # LIVE DISPLAY
    # ========================================================
    image_handle = display(display_id=True)

    # ========================================================
    # LIVE LOOP
    # ========================================================
    try:
        while True:
            try:
                active = eval_js("window.isWebcamActive()")
            except Exception:
                active = False
            if not active:
                break

            frame_data = eval_js("window.captureFrame()")
            if not frame_data:
                time.sleep(0.1)
                continue

            try:
                binary = b64decode(frame_data.split(",", 1)[1])
                image = Image.open(io.BytesIO(binary)).convert("RGB")
            except Exception:
                continue

            # =================================================
            # SOIL PREDICTION
            # =================================================
            input_tensor = soil_transform(image).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                output = soil_model(input_tensor)
                probabilities = torch.softmax(output, dim=1)[0]
                confidence, index = torch.max(probabilities, dim=0)

            soil_name = SOIL_CLASSES[index.item()]
            confidence_percent = confidence.item() * 100

            # =================================================
            # UPDATE SESSION DATA
            # =================================================
            total_frames += 1
            soil_counts[soil_name] = soil_counts.get(soil_name, 0) + 1

            if soil_name not in soil_confidences:
                soil_confidences[soil_name] = []
            soil_confidences[soil_name].append(confidence_percent)
            confidence_values.append(confidence_percent)

            # =================================================
            # DRAW LIVE RESULT
            # =================================================
            result_image = image.copy()
            draw = ImageDraw.Draw(result_image)
            draw.rounded_rectangle([15, 15, 440, 105], radius=15, fill=(0, 0, 0))
            draw.text((30, 30), f"Soil: {soil_name}", fill=(255, 255, 255))
            draw.text((30, 65), f"Confidence: {confidence_percent:.2f}%", fill=(255, 255, 255))

            # =================================================
            # UPDATE DISPLAY
            # =================================================
            image_handle.update(result_image)
            time.sleep(0.15)

    except KeyboardInterrupt:
        print("Webcam stopped by user.")

    finally:
        # ====================================================
        # STOP CAMERA
        # ====================================================
        try:
            eval_js("window.stopWebcam()")
        except:
            pass

        # ====================================================
        # FINAL SUMMARY REPORT
        # ====================================================
        IPython.display.clear_output(wait=True)

        if total_frames == 0:
            print("No frames were analyzed. No report generated.")
        else:
            # Build records
            report_records = []
            for s_name, count in soil_counts.items():
                confs = soil_confidences[s_name]
                report_records.append({
                    "Soil Type": s_name,
                    "Detection Count": count,
                    "Average Confidence": round(sum(confs) / len(confs), 2),
                    "Highest Confidence": round(max(confs), 2)
                })

            df_report = pd.DataFrame(report_records)
            df_report = df_report.sort_values(by="Detection Count", ascending=False).reset_index(drop=True)

            most_detected_soil = df_report.iloc[0]["Soil Type"]
            most_detected_count = int(df_report.iloc[0]["Detection Count"])
            overall_confidence = sum(confidence_values) / len(confidence_values)

            # =================================================
            # EXTREMELY BEAUTIFUL HTML REPORT
            # =================================================
            rows_html = ""
            for i, row in df_report.iterrows():
                bg_color = "#fdfdfd" if i % 2 == 0 else "#f5f9f5"
                rows_html += f"""
                <tr style="background-color: {bg_color}; border-bottom: 1px solid #e2ece2; transition: 0.2s;">
                    <td style="padding: 16px 20px; font-weight: 600; color: #1e3a1e; font-size: 15px;">
                        🌱 {row['Soil Type']}
                    </td>
                    <td style="padding: 16px 20px; text-align: center; color: #2e4d2e; font-weight: 700; font-size: 15px;">
                        {int(row['Detection Count'])}
                    </td>
                    <td style="padding: 16px 20px; text-align: center; color: #1b5e20; font-weight: 600; font-size: 15px;">
                        {row['Average Confidence']:.2f}%
                    </td>
                    <td style="padding: 16px 20px; text-align: center; color: #0d47a1; font-weight: 600; font-size: 15px;">
                        {row['Highest Confidence']:.2f}%
                    </td>
                </tr>
                """

            report_html = f"""
            <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700;800&display=swap" rel="stylesheet">
            <div style="
                max-width: 750px;
                margin: 30px auto;
                padding: 40px;
                background: #ffffff;
                border-radius: 24px;
                box-shadow: 0 15px 45px rgba(46, 125, 50, 0.08), 0 4px 12px rgba(0,0,0,0.02);
                font-family: 'Plus Jakarta Sans', system-ui, sans-serif;
                border: 1px solid rgba(46, 125, 50, 0.1);
            ">

                <!-- HEADER -->
                <div style="text-align: center; margin-bottom: 35px;">
                    <span style="
                        background: rgba(46, 125, 50, 0.08);
                        color: #2e7d32;
                        padding: 6px 16px;
                        border-radius: 100px;
                        font-size: 12px;
                        font-weight: 700;
                        letter-spacing: 1.5px;
                        text-transform: uppercase;
                    ">COMPUTER VISION ANALYSIS</span>
                    <h2 style="
                        margin: 15px 0 5px 0;
                        font-size: 32px;
                        font-weight: 800;
                        color: #1b381e;
                        letter-spacing: -0.5px;
                    ">Soil Detection Report</h2>
                    <p style="color: #667e68; font-size: 15px; margin: 0;">Real-time agricultural environment classification metrics</p>
                </div>

                <!-- METRIC CARDS -->
                <div style="
                    display: grid;
                    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                    gap: 16px;
                    margin-bottom: 35px;
                ">
                    <div style="
                        background: linear-gradient(135deg, #f1f8e9, #e8f5e9);
                        border-radius: 16px;
                        padding: 20px;
                        text-align: center;
                        border: 1px solid rgba(139, 195, 74, 0.2);
                    ">
                        <div style="font-size: 11px; font-weight: 700; color: #558b2f; letter-spacing: 0.5px; text-transform: uppercase;">Dominant Soil</div>
                        <div style="font-size: 20px; font-weight: 800; color: #2e7d32; margin-top: 8px;">{most_detected_soil}</div>
                    </div>

                    <div style="
                        background: #fafafa;
                        border-radius: 16px;
                        padding: 20px;
                        text-align: center;
                        border: 1px solid #eeeeee;
                    ">
                        <div style="font-size: 11px; font-weight: 700; color: #616161; letter-spacing: 0.5px; text-transform: uppercase;">Analyzed Frames</div>
                        <div style="font-size: 24px; font-weight: 800; color: #212121; margin-top: 4px;">{total_frames}</div>
                    </div>

                    <div style="
                        background: linear-gradient(135deg, #e3f2fd, #e1f5fe);
                        border-radius: 16px;
                        padding: 20px;
                        text-align: center;
                        border: 1px solid rgba(33, 150, 243, 0.15);
                    ">
                        <div style="font-size: 11px; font-weight: 700; color: #1565c0; letter-spacing: 0.5px; text-transform: uppercase;">Avg Accuracy</div>
                        <div style="font-size: 24px; font-weight: 800; color: #0d47a1; margin-top: 4px;">{overall_confidence:.2f}%</div>
                    </div>
                </div>

                <!-- DATA TABLE -->
                <div style="overflow-hidden; border-radius: 16px; border: 1px solid #e2ece2; margin-bottom: 25px;">
                    <table style="
                        width: 100%;
                        border-collapse: collapse;
                        text-align: left;
                    ">
                        <thead>
                            <tr style="background: #eaf2ea; border-bottom: 2px solid #c8dbc8;">
                                <th style="padding: 16px 20px; font-size: 12px; font-weight: 800; color: #2e4d2e; text-transform: uppercase; letter-spacing: 0.5px;">Soil Type</th>
                                <th style="padding: 16px 20px; font-size: 12px; font-weight: 800; color: #2e4d2e; text-transform: uppercase; letter-spacing: 0.5px; text-align: center;">Detections</th>
                                <th style="padding: 16px 20px; font-size: 12px; font-weight: 800; color: #2e4d2e; text-transform: uppercase; letter-spacing: 0.5px; text-align: center;">Avg Confidence</th>
                                <th style="padding: 16px 20px; font-size: 12px; font-weight: 800; color: #2e4d2e; text-transform: uppercase; letter-spacing: 0.5px; text-align: center;">Highest Confidence</th>
                            </tr>
                        </thead>
                        <tbody>
                            {rows_html}
                        </tbody>
                    </table>
                </div>

                <!-- SUMMARY BADGE FOOTER -->
                <div style="
                    padding: 18px 24px;
                    background: linear-gradient(135deg, #2e7d32, #1b5e20);
                    border-radius: 16px;
                    text-align: center;
                    color: #ffffff;
                    font-weight: 600;
                    font-size: 15px;
                    box-shadow: 0 8px 20px rgba(46,125,50,0.15);
                ">
                    ✨ High confidence primary prediction identifies <span style="font-weight: 800; text-decoration: underline;">{most_detected_soil}</span> with {most_detected_count} distinct keyframes.
                </div>
            </div>
            """

            display(HTML(report_html))
            print("\nDetailed Report DataFrame Summary:")
            display(df_report)


RuntimeError: ❌ soil_transform is not loaded.